# CDC-CP P1e: does patch shuffle help unseen disease?

The teacher's tip: cut the training picture into patches and mix them, so the model reads the disease spots and not the leaf shape.

| Arm | Config | Change | Seeds |
|---|---|---|---|
| Control | `configs/p1b_base.yaml` | none | 42, 43, 44 |
| Patch shuffle | `configs/p1b_patch.yaml` | half the training images cut into a 4 x 4 grid and mixed | 42, 43, 44 |

Both arms: EfficientNet-B0, backbone LR 1e-4, heads 1e-3, batch 32, at most 12 epochs, **early stopping with patience 3** on unseen disease Top-1 (`scripts/train_patience.py`). Only the patch shuffle and the seed change.

**Rule (`cdccp/selection.py: compare_arms`, unit-tested).** Each run scores its best full 3-epoch average of unseen disease Top-1. The two arms are compared seed by seed. Patch shuffle is **better** or **worse** only when the mean difference is more than 2 standard errors (seed spread plus the noise of 400 unseen images, about 5-6 points). Otherwise the result is **no clear change**. A trivial row (always guess the most common training disease) is shown so the numbers can be read against it.

Early stopping and the score both pick on the same validation images, so the numbers are a little too high, and a run that trains longer gets a few more tries. This is a development comparison of two recipes. It is not a significance test and it does not estimate the official test.

**Setup (once):** inputs are the code dataset (the new `cdc-cp_code_p1e.zip`) and the data dataset, as before. **Accelerator: GPU T4 x2**, **Internet: On**.

**First, a smoke test:** keep `SMOKE = True` and run all. It trains 2 epochs of each arm and prints the minutes per run. Then set `SMOKE = False` and **Save Version, Save & Run All (Commit)**.

**If it stops early:** add this notebook's last version (its output) as an input, then Save & Run All again. Finished runs are skipped and an interrupted run resumes.

Only `DPD_Small_train.csv` and its 8,000 images are read. The official test files are never opened.

## 1. Settings

In [ ]:
from pathlib import Path

SMOKE    = True    # True: 2 epochs of each arm (seed 42) to check it runs and to time it. Then set False and commit.
WORKERS  = 2       # data-loader workers per run (2 runs share Kaggle's 4 CPU cores)
PATIENCE = 3       # stop after this many epochs without a new best unseen disease Top-1

CONTROL, TREATMENT = 'p1b_base', 'p1b_patch'
ARMS  = [CONTROL, TREATMENT]
SEEDS = (42, 43, 44)

INPUT     = Path('/kaggle/input')
RUNS_DIR  = Path('/kaggle/working/p1e_smoke' if SMOKE else '/kaggle/working/runs')
CODE_DIR  = Path('/tmp/cdc-cp')
DATA_DIR  = Path('/tmp/DPD_Small')
CACHE_DIR = Path('/tmp/cache')

## 2. GPUs and code

In [ ]:
import os, shutil, subprocess, sys, zipfile, time
import torch

assert torch.cuda.is_available(), 'Settings > Accelerator > GPU T4 x2'
N_GPU = torch.cuda.device_count()
print(N_GPU, 'x', torch.cuda.get_device_name(0), '| torch', torch.__version__)
if N_GPU < 2:
    print('Only one GPU: runs will go one at a time (about twice as long). Pick GPU T4 x2 to run two at once.')

os.chdir('/kaggle/working')  # a rerun deletes CODE_DIR, so the kernel must not be inside it
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
found = next((p.parent.parent for p in INPUT.rglob('cdccp/__init__.py')), None)
if found is not None:
    shutil.copytree(found, CODE_DIR)
else:
    code_zip = next(INPUT.rglob('cdc-cp_code*.zip'), None)
    assert code_zip, 'Add the code dataset (cdc-cp_code.zip) as an input'
    with zipfile.ZipFile(code_zip) as z:
        z.extractall('/tmp')
assert (CODE_DIR / 'scripts' / 'train_patience.py').is_file() and (CODE_DIR / 'configs' / 'p1b_patch.yaml').is_file(), \
    'old code dataset: upload the new cdc-cp_code_p1e.zip'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)
os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
from cdccp.selection import compare_arms, run_scores
print('code from', found or code_zip)


def sh(*args):
    proc = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'exit code {proc.returncode}: {args}')

## 3. Training data (train CSV and its 8,000 images only)

In [ ]:
import hashlib
import pandas as pd

train_csv = next(p for p in INPUT.rglob('*') if p.name.lower() == 'dpd_small_train.csv')
cols = ['filename', 'crop_id', 'disease_id', 'composition_id', 'crop_name', 'disease_name', 'composition_name']
train = pd.read_csv(train_csv, header=None, names=cols)
assert len(train) == 8000, f'expected 8,000 rows, got {len(train)}'
digest = hashlib.sha256(train_csv.read_bytes()).hexdigest()
assert digest.startswith('9f1b729a'), f'train CSV hash {digest[:8]} does not match the split manifest'

first = train.filename.iloc[0]
image_src = next((d for d in [train_csv.parent, *train_csv.parent.parent.rglob('*')] if d.is_dir() and (d / first).is_file()), None)
assert image_src is not None, f'could not find {first} near {train_csv.parent}'
DATA_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(train_csv, DATA_DIR / 'DPD_Small_train.csv')
link = DATA_DIR / 'DPD_Small_dataset'
if not link.exists():
    link.symlink_to(image_src, target_is_directory=True)
missing = [n for n in train.filename if not (link / n).is_file()]
assert not missing, f'{len(missing)} images missing, e.g. {missing[:3]}'
os.environ['DPD_ROOT'] = str(DATA_DIR)
print(f'8,000 training images linked from {image_src}')

## 4. Restore runs from earlier outputs

In [ ]:
from cdccp.config import load_config
from cdccp.train import _recipe

RUNS_DIR.mkdir(parents=True, exist_ok=True)


def run_name(arm, seed):
    return f'{arm}_seed{seed}'


def overrides(seed):
    return [f'paths.runs_dir={RUNS_DIR}', f'paths.cache_dir={CACHE_DIR}', f'loader.num_workers={WORKERS}', f'seed={seed}'] + (['epochs=2'] if SMOKE else [])


def progress(run_dir):
    # (finished, applied updates) of a run folder; a run with early_stop.json is finished
    return (run_dir / 'early_stop.json').is_file(), torch.load(run_dir / 'last.pt', map_location='cpu', weights_only=False)['applied']


# Only copy a run whose recipe equals the one this notebook would train, so a smoke run or another config is never mixed in.
restored = []
if not SMOKE:
    for arm in ARMS:
        for seed in SEEDS:
            want = _recipe(load_config(CODE_DIR / 'configs' / f'{arm}.yaml', overrides(seed)))
            dst = RUNS_DIR / run_name(arm, seed)
            for last in INPUT.rglob(f'{run_name(arm, seed)}/last.pt'):
                if not (last.parent / 'patience_start.json').is_file():
                    print('skip (not an early-stopping run):', last.parent)
                    continue
                if torch.load(last, map_location='cpu', weights_only=False)['recipe'] != want:
                    print('skip (other recipe):', last.parent)
                    continue
                if not (dst / 'last.pt').exists() or progress(dst) < progress(last.parent):
                    shutil.rmtree(dst, ignore_errors=True)
                    shutil.copytree(last.parent, dst)
                    restored.append(str(last.parent))
print('restored:', restored or 'nothing')

## 5. Image cache (local disk, a few minutes)

In [ ]:
sh(sys.executable, 'scripts/build_cache.py', '--verify', '--workers', 8, f'paths.cache_dir={CACHE_DIR}')

## 6. Run queue
Each GPU takes the next waiting run. A run keeps its last 3 epoch checkpoints and its best one.

In [ ]:
import json


def job(arm, seed):
    return run_name(arm, seed), [sys.executable, 'scripts/train_patience.py', f'configs/{arm}.yaml', *overrides(seed),
                                 '--patience', PATIENCE, '--resume']


def prune(run_dir, keep=3):
    # keep the last `keep` epoch checkpoints, the best epoch's and the scored epoch's (centre of the best 3-epoch window)
    scores = run_scores(pd.read_csv(run_dir / 'history.csv')) if (run_dir / 'history.csv').is_file() else {}
    spare = {f'epoch_{scores[k]:02d}.pt' for k in ('best_epoch', 'pick_epoch') if k in scores}
    for p in sorted(run_dir.glob('epoch_*.pt'))[:-keep]:
        if p.name not in spare:
            p.unlink()


def run_queue(jobs):
    """Run (name, args) jobs, one per GPU at a time. A failed run is reported and left out of the ranking."""
    waiting, running, failed = list(jobs), {}, []
    print('queue:', [n for n, _ in waiting], flush=True)
    while waiting or running:
        for gpu in range(max(N_GPU, 1)):
            if gpu not in running and waiting:
                name, args = waiting.pop(0)
                log = open(RUNS_DIR / f'{name}.log', 'a')
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
                p = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, env=env, stdout=log, stderr=subprocess.STDOUT, text=True)
                running[gpu] = (name, p, log, 0, time.time())
                print(f'[gpu {gpu}] start {name}', flush=True)
        time.sleep(20)
        for gpu, (name, p, log, seen, t0) in list(running.items()):
            log.flush()
            lines = (RUNS_DIR / f'{name}.log').read_text(errors='replace').splitlines()
            for line in lines[seen:]:
                if line.startswith(('epoch', 'done', 'early stop', 'nothing to do', 'Traceback')) or 'Error' in line:
                    print(f'[gpu {gpu}] {name}: {line}', flush=True)
            running[gpu] = (name, p, log, len(lines), t0)
            if p.poll() is not None:
                log.close()
                del running[gpu]
                mins = (time.time() - t0) / 60
                if p.returncode:
                    failed.append(name)
                    print(f'[gpu {gpu}] FAILED {name} after {mins:.0f} min; last lines:', *lines[-15:], sep='\n', flush=True)
                else:
                    print(f'[gpu {gpu}] finished {name} in {mins:.0f} min', flush=True)
                    if (RUNS_DIR / name).is_dir():
                        prune(RUNS_DIR / name)
    if failed:
        print(f'WARNING failed runs, left out of the ranking: {failed} (logs in {RUNS_DIR})', flush=True)
    return failed

## 7. Train 2 arms x 3 seeds (early stopping, patience 3)

In [ ]:
FAILED = run_queue([job(a, s) for s in ((42,) if SMOKE else SEEDS) for a in ARMS])
if SMOKE:
    assert not FAILED, f'smoke runs failed: {FAILED}. Read their logs in {RUNS_DIR} before the full run.'
    for a in ARMS:
        print(a, (RUNS_DIR / run_name(a, 42) / 'early_stop.json').read_text()[:120])
    print('SMOKE run finished. Each full run is at most 6 times the minutes above (12 epochs, not 2).')
    print('Set SMOKE = False and Save & Run All (Commit).')

## 8. Compare
Saved as `runs/patch_compare.md` and `runs/patch_compare_runs.csv` in this version's output.

In [ ]:
from cdccp.data import load_split

if not SMOKE:
    if FAILED:
        verdict = (f'NO VERDICT: runs failed: {FAILED}. Read their logs above. Then add this version as an input and Save & Run All again: '
                   'finished runs are skipped, failed ones resume.')
    else:
        per_run = pd.DataFrame([{'arm': a, 'seed': s, **run_scores(pd.read_csv(RUNS_DIR / run_name(a, s) / 'history.csv'))}
                                for a in ARMS for s in SEEDS])
        table, diffs, mean, limit, word = compare_arms(per_run, CONTROL, TREATMENT, SEEDS)
        split_cfg = load_config(CODE_DIR / 'configs' / f'{CONTROL}.yaml', overrides(42))
        fit, unseen = load_split(split_cfg, 'train_fit'), load_split(split_cfg, 'val_unseen')
        trivial = 100 * (unseen.disease_id == fit.disease_id.mode()[0]).mean()
        show = table.rename(columns={'unseen_disease': 'unseen disease (mean)', 'sd': 'SD over seeds', 'seen_disease': 'seen disease'}).round(2)
        show.loc['trivial: always the most common training disease'] = [round(trivial, 2), float('nan'), float('nan')]
        lines = [per_run.round(2).to_string(index=False), '', show.to_string(), '',
                 'patch minus control, per seed: ' + ', '.join(f'{s}: {d:+.1f}' for s, d in diffs.items())]
        text = '\n'.join(lines)
        print(text)
        verdict = {'better': f'BETTER: patch shuffle adds {mean:+.1f} points of unseen disease Top-1 (the bar is {limit:.1f}).',
                   'worse': f'WORSE: patch shuffle changes unseen disease Top-1 by {mean:+.1f} points (the bar is {limit:.1f}).',
                   'no clear change': (f'NO CLEAR CHANGE: patch shuffle changes unseen disease Top-1 by {mean:+.1f} points; the bar is '
                                       f'{limit:.1f} (seed spread plus the noise of 400 unseen images).')}[word]
        verdict = ('```\n' + text + '\n```\n\n' + verdict + '\n\nThis compares two training recipes on development validation. '
                   'Early stopping and the score pick on the same images, so it is not a significance test and not a test-set estimate.')
        per_run.to_csv(RUNS_DIR / 'patch_compare_runs.csv', index=False)
    print(verdict)
    (RUNS_DIR / 'patch_compare.md').write_text('# P1e patch shuffle vs control (development validation)\n\n' + verdict + '\n', encoding='utf-8')

## 9. Curves

In [ ]:
if not SMOKE and not FAILED:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    colours = {CONTROL: 'tab:blue', TREATMENT: 'tab:orange'}
    for a in ARMS:
        for s in SEEDS:
            h = pd.read_csv(RUNS_DIR / run_name(a, s) / 'history.csv')
            label = f'{a} s{s}'
            axes[0].plot(h.epoch, h.unseen_disease_top1, color=colours[a], label=label, marker='.')
            axes[1].plot(h.epoch, h.seen_disease_top1, color=colours[a], label=label, marker='.')
            axes[2].plot(h.epoch, h.val_ce_disease, color=colours[a], label=label, marker='.')
    for ax, t in zip(axes, ['Unseen disease Top-1', 'Seen disease Top-1', 'Validation disease cross-entropy (seen)']):
        ax.set_title(t); ax.set_xlabel('epoch (a line ends where early stopping stopped it)'); ax.grid(alpha=.3); ax.legend(fontsize=6)
    plt.tight_layout(); plt.show()